# Model comparison for Thrust and Damping forces and moments calculations (No standardization)

For this model the usage of a flexible FNN is explored. With this structure the number of parameters can be changed in runtime. The option to enable/disable dropout is also included.

In this case, the input and output are scaled using standardization. This experiment served to asses the effect of the standardization of the data in the performance of the NN.

In this case the data was generated by varying the tail deflection and tracking the following variables from the MATLAB simulation:

Input: xout, cout (caudal fin amplitude/angle), caudal_amp_old, Tail_Centre_out
Output: Tail_Forces_out.

Additionally, the variables that are constant and zero, were excluded from the target data(output)


In [1]:
num_epochs = 10
neurons_per_layer = [16, 32, 64]
hidden_layers = [5]
model_prefix = "simple_data_modelv2_"

In [1]:
import numpy as np
import nn_fncs
mat_data = nn_fncs.read_mat_workspace('Thrust_data.mat')
nn_in = mat_data.get('nn_in')
nn_out = mat_data.get('nn_out')
# Get every 5th data sample
nn_in = nn_in[:, ::5, :]
nn_out = nn_out[:, ::5, :]
# Print shapes
print(f'Original nn_in shape: {nn_in.shape}')  # (num_trajectories, num_time_steps, num_inputs)
print(f'Original nn_out shape: {nn_out.shape}')  # (num_trajectories, num_time_steps, num_outputs)
# remove zero columns in nn_out
nn_out1 = nn_out[:, :, ~np.all(nn_out == 0, axis=(0, 1))]
nn_out2 = nn_out[:, :, [0, 1, 2, 3, 7, 10, 11]] 
# Verify that all elements of nn_out1 and nn_out2 are the same
assert np.array_equal(nn_out1, nn_out2), "The arrays are not equal"
nn_out = nn_out2
print(f'nn_in shape: {nn_in.shape}')  # (num_trajectories, num_time_steps, num_inputs)
print(f'nn_out shape: {nn_out.shape}')      # (num_trajectories, num_time_steps, num_outputs)
# Reshape to 2D arrays for training
num_trajectories, num_time_steps, num_inputs = nn_in.shape
num_outputs = nn_out.shape[2]
nn_in = nn_in.reshape(-1, num_inputs)
nn_out = nn_out.reshape(-1, num_outputs)
print(f'Reshaped nn_in shape: {nn_in.shape}')  # (num_trajectories * num_time_steps, num_inputs)
print(f'Reshaped nn_out shape: {nn_out.shape}')  # (num_

Original nn_in shape: (329, 2000, 16)
Original nn_out shape: (329, 2000, 12)
nn_in shape: (329, 2000, 16)
nn_out shape: (329, 2000, 7)
Reshaped nn_in shape: (658000, 16)
Reshaped nn_out shape: (658000, 7)


In [3]:
# Create data loaders
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
# Split into training and validation sets (80% train, 20% val)
# separate into training and validation sets
split_ratio = 0.75
split_index = int(nn_in.shape[0] * split_ratio)
# Shuffle data before splitting
indices = np.arange(nn_in.shape[0])
np.random.shuffle(indices)
nn_in = nn_in[indices]
nn_out = nn_out[indices]

# Split data and then standardize based on training data
in_train = nn_in[:split_index]
in_valid = nn_in[split_index:]
scaler_in = StandardScaler()
scaler_in.fit(in_train)
in_train = scaler_in.transform(in_train)
in_valid = scaler_in.transform(in_valid)
# Split and standardize outputs
nn_out_train = nn_out[:split_index]
nn_out_valid = nn_out[split_index:]
scaler_out = StandardScaler()
scaler_out.fit(nn_out_train)
nn_out_train = scaler_out.transform(nn_out_train)
nn_out_valid = scaler_out.transform(nn_out_valid)

#Create DataLoader objects
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# Split data into training and validation sets
train_dataset = torch.utils.data.TensorDataset(torch.tensor(in_train, dtype=torch.float32).to(device), 
                                               torch.tensor(nn_out_train, dtype=torch.float32).to(device))
valid_dataset = torch.utils.data.TensorDataset(torch.tensor(in_valid, dtype=torch.float32).to(device), 
                                               torch.tensor(nn_out_valid, dtype=torch.float32).to(device))

train_loader = DataLoader(train_dataset, batch_size=1000, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=1000, shuffle=False)

In [4]:
# MODEL STRUCTURE

import torch
import torch.nn as nn
import torch.nn.functional as F

# Function to stack n layers
import torch
import torch.nn as nn
import torch.nn.functional as F

class thrustFlexNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, hidden_layers=3, dropout_enabled=True):
        super(thrustFlexNN, self).__init__()
        self.dropout_enabled = dropout_enabled
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(hidden_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        self.layers.append(nn.Linear(hidden_size, output_size))

    def forward(self, x):
        out = x
        for layer in self.layers[:-1]:
            out = F.relu(layer(out))
            if self.dropout_enabled:
                out = F.dropout(out, p=0.1)
        out = self.layers[-1](out)
        return out
    


# model = ThrustModel(nn_in.shape[1], nn_out.shape[1])

# # Print model summary
# print(model)

# # Count trainable parameters
# total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f'Total trainable parameters: {total_params}')

In [5]:
# One step training
import torch.optim as optim
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_step_training(model, criterion, optimizer, input_data, target, r2_scalar = True):
    t = 0
    loss = 0.0
    xk = target[0]
    predictions = torch.zeros_like(target)
    for i in range(target.shape[0]-1):

        with torch.set_grad_enabled(True):
            # Forward pass
            pred = model(input_data)  # Integrate over a small time step
            loss = criterion(pred, target)
            # Backward and optimize
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    if r2_scalar:
        r2 = r2_score(pred, target, multioutput='uniform_average')
    else:
        r2 = r2_score(pred, target, multioutput='raw_values')
    return pred, loss, r2

In [6]:
# One step eval
import torch.optim as optim
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_eval_step(model, input_data, target, r2_multiout=False):
    loss = 0.0
    model.eval()

    for i in range(target.shape[0]-1):

        with torch.no_grad():
            # Forward pass
            pred = model(input_data)  # Integrate over a small time step
    mse = mean_squared_error(pred, target)
            # Backward and optimize
    if r2_multiout:
        r2 = r2_score(pred, target, multioutput='raw_values')
    else:    
        r2 = r2_score(pred, target)
    return pred, mse, r2

In [7]:
# Complete training loop
import torch
from tqdm import tqdm

def model_training_loop(model, train_loader, valid_loader, num_epochs=1000, epoch_update=10, model_name='thrust_model'):
    best_r2 = -float('inf')
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    history_train = {'loss': [], 'r2': []}
    history_val = {'val_loss': [], 'val_r2': []}
    with tqdm(total=num_epochs) as pbar:
        for epoch in range(num_epochs):
            epoch_loss = 0.0
            epoch_r2 = 0.0
            for in_tensor, out_tensor in train_loader:
                pred, loss, r2 = one_step_training(model, 
                                                    criterion, 
                                                    optimizer,
                                                    in_tensor,
                                                    out_tensor)
                epoch_loss += loss.item()*in_tensor.size(0)
                epoch_r2 += r2.item()*in_tensor.size(0)
            epoch_loss /= (train_loader.dataset.tensors[0].shape[0])
            epoch_r2 /= (train_loader.dataset.tensors[1].shape[0])
            history_train['loss'].append(epoch_loss)
            history_train['r2'].append(epoch_r2)
            if (epoch+1) % epoch_update == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6e}, R2: {epoch_r2:.6e}')

            val_loss = 0.0
            val_r2 = 0.0
            for in_tensor, out_tensor in valid_loader:
                pred, loss, r2 = one_eval_step(model, in_tensor, out_tensor)
                val_loss += loss*in_tensor.size(0)
                val_r2 += r2.item()*in_tensor.size(0)
            val_loss /= (valid_loader.dataset.tensors[0].shape[0])
            val_r2 /= (valid_loader.dataset.tensors[1].shape[0])
            if (epoch+1) % epoch_update == 0:
                print(f'Validation Loss: {val_loss:.6e}, Validation R2: {val_r2:.6e}')
                pbar.update(epoch_update)
            if val_r2 > best_r2:
                best_r2 = val_r2
                nn_fncs.save_best_model(model, val_r2, 0, model_name=model_name)
            history_val['val_loss'].append(val_loss)
            history_val['val_r2'].append(val_r2)
    return history_val, history_train, best_r2


In [ ]:
modebest_r2 = -float('inf')
for npl in neurons_per_layer:
    for hl in hidden_layers:
        model_name = model_prefix + f'{npl}_neurons_{hl}_layers_'
        print(f'Training model with {npl} neurons per layer and {hl} hidden layers')
        model = thrustFlexNN(input_size=nn_in.shape[1], hidden_size=npl, output_size=nn_out.shape[1], hidden_layers=hl, dropout_enabled=True)
        history_val, history_train, best_r2 = model_training_loop(model, train_loader, valid_loader, num_epochs=num_epochs, epoch_update=1, model_name=model_name)
        

Training model with 16 neurons per layer and 5 hidden layers


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch [1/10], Loss: 1.162356e-01, R2: 8.728587e-01


 10%|█         | 1/10 [31:05<4:39:49, 1865.51s/it]

Validation Loss: 1.544700e-01, Validation R2: 8.226061e-01
New best model saved: simple_data_modelv2_16_neurons_5_layers__0.8226061489.pt (Accuracy: 0.8226061489)
Epoch [2/10], Loss: 1.123812e-01, R2: 8.774137e-01


...

Validation Loss: 1.047001e-01, Validation R2: 8.853111e-01
Epoch [9/10], Loss: 3.348833e-02, R2: 9.632479e-01


 90%|█████████ | 9/10 [3:45:29<22:45, 1365.25s/it]

Validation Loss: 1.184129e-01, Validation R2: 8.653614e-01
Epoch [10/10], Loss: 3.337479e-02, R2: 9.638560e-01


100%|██████████| 10/10 [4:07:08<00:00, 1482.83s/it]

Validation Loss: 1.167065e-01, Validation R2: 8.488509e-01


In [10]:
# Save Scaler objects
import joblib

joblib.dump(scaler_in, 'scaler_thrust_in.pkl')
joblib.dump(scaler_out, 'scaler_trust_out.pkl')

['scaler_trust_out.pkl']

In [13]:
num_epochs = 10
neurons_per_layer = [4, 8, 16]
hidden_layers = [3, 5]
model_prefix = "no_dropout"
modebest_r2 = -float('inf')
for npl in neurons_per_layer:
    for hl in hidden_layers:
        model_name = model_prefix + f'{npl}_neurons_{hl}_layers_'
        print(f'Training model with {npl} neurons per layer and {hl} hidden layers')
        model = thrustFlexNN(input_size=nn_in.shape[1], hidden_size=npl, output_size=nn_out.shape[1], hidden_layers=hl, dropout_enabled=False)
        history_val, history_train, best_r2 = model_training_loop(model, train_loader, valid_loader, num_epochs=num_epochs, epoch_update=1, model_name=model_name)
        

Training model with 4 neurons per layer and 3 hidden layers


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch [1/10], Loss: 1.677820e-01, R2: 8.045563e-01


 10%|█         | 1/10 [14:33<2:11:00, 873.41s/it]

Validation Loss: 2.341340e-01, Validation R2: 7.293424e-01
New best model saved: no_dropout4_neurons_3_layers__0.7293424490.pt (Accuracy: 0.7293424490)
Epoch [2/10], Loss: 1.684990e-01, R2: 8.105580e-01


...

Validation Loss: 1.586161e-02, Validation R2: 9.802616e-01
Epoch [9/10], Loss: 1.850207e-03, R2: 9.975492e-01


 90%|█████████ | 9/10 [12:34:42<1:07:32, 4052.10s/it]

Validation Loss: 1.740113e-02, Validation R2: 9.782244e-01
Epoch [10/10], Loss: 1.806334e-03, R2: 9.974928e-01


100%|██████████| 10/10 [12:54:40<00:00, 4648.06s/it] 

Validation Loss: 1.350678e-02, Validation R2: 9.827112e-01
